In [3]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import os

In [4]:
def generate_random_conic_type():
    """Возвращает случайный тип кривой: эллипс, парабола или гипербола."""
    return np.random.choice(["ellipse", "parabola", "hyperbola"])

def generate_ellipse_data(noise_std, N=1000, x_range=(0,100)):
    # Генерация параметров эллипса: центр (cx, cy), полуоси (a_axis, b_axis) и угол поворота theta.
    cx = np.random.uniform(30, 70)
    cy = np.random.uniform(30, 70)
    a_axis = np.random.uniform(10, 30)
    b_axis = np.random.uniform(10, 30)
    theta = np.random.uniform(0, np.pi)
    
    t = np.linspace(0, 2*np.pi, N)
    x_clean = cx + a_axis * np.cos(t)*np.cos(theta) - b_axis * np.sin(t)*np.sin(theta)
    y_clean = cy + a_axis * np.cos(t)*np.sin(theta) + b_axis * np.sin(t)*np.cos(theta)
    
    x_noisy = x_clean + np.random.normal(0, noise_std, N)
    y_noisy = y_clean + np.random.normal(0, noise_std, N)
    
    # Возвращаем данные и параметры, по которым строился эллипс.
    params = (cx, cy, a_axis, b_axis, theta)
    return x_clean, y_clean, x_noisy, y_noisy, params

def generate_parabola_data(noise_std, N=1000, x_range=(0,100)):
    # Парабола задаётся уравнением: y = A*(x-h)^2 + k.
    h = np.random.uniform(30, 70)
    k = np.random.uniform(0, 100)
    A = np.random.uniform(-0.1, 0.1)
    
    x_clean = np.linspace(x_range[0], x_range[1], N)
    y_clean = A*(x_clean - h)**2 + k
    x_noisy = x_clean + np.random.normal(0, noise_std, N)
    y_noisy = y_clean + np.random.normal(0, noise_std, N)
    
    params = (h, k, A)
    return x_clean, y_clean, x_noisy, y_noisy, params

def generate_hyperbola_data(noise_std, N=1000, x_range=(0,100)):
    # Случайные параметры для гиперболы
    h = np.random.uniform(30, 70)
    k = np.random.uniform(30, 70)
    a = np.random.uniform(10, 20)
    b = np.random.uniform(10, 20)
    
    # Начальная точка гиперболы: x0 = h + a.
    x0 = h + a
    
    # Определяем t_max так, чтобы x не превышало x_range[1].
    # Если (x_range[1]-h)/a < 1, значит, заданный диапазон слишком узкий – зададим t_max=0.1.
    ratio = (x_range[1] - h) / a
    if ratio < 1:
        t_max = 0.1
    else:
        t_max = np.arccosh(ratio)
    
    # Для более красивого вида можно добавить небольшой запас по t:
    t_max = t_max + 0.5
    
    t = np.linspace(-t_max, t_max, N)
    x_clean = h + a * np.cosh(t)
    y_clean = k + b * np.sinh(t)
    
    # Добавляем шум к x и y
    x_noisy = x_clean + np.random.normal(0, noise_std, N)
    y_noisy = y_clean + np.random.normal(0, noise_std, N)
    
    params = (h, k, a, b)
    return x_clean, y_clean, x_noisy, y_noisy, params


def save_data_to_csv(X, Y, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['x', 'y'])
        for xi, yi in zip(X, Y):
            writer.writerow([xi, yi])


def generate_outliers(num_outliers, center_x, center_y, std):
    x_out = np.random.normal(center_x, std, num_outliers)
    y_out = np.random.normal(center_y, std, num_outliers)
    return x_out, y_out

def generate_linear_outlier_line_params(x_range=(0,100)):
    a_out = np.random.uniform(-3.0, 3.0)       
    b_out = np.random.uniform(-20.0, 20.0)        
    return a_out, b_out

def generate_linear_outliers(num_points, a_out, b_out, x_range=(0,100), noise_std=2.0):
    x_out = np.linspace(x_range[0] + 5, x_range[1] - 5, num_points)
    y_line = a_out * x_out + b_out
    
    y_out = y_line + np.random.normal(0, noise_std, num_points)
    return x_out, y_out

def generate_random_noise_outliers(num_points, x_range, y_range):
    x_out = np.random.uniform(x_range[0], x_range[1], num_points)
    y_out = np.random.uniform(y_range[0], y_range[1], num_points)
    return x_out, y_out

In [ ]:
def gen_conic_outlier(n_out, k, category = "stat"):
    """
    Генерирует данные для конической кривой (ellipse, parabola или hyperbola)
    с аддитивным шумом и выбросами. Тип выбросов определяется параметром category:
      - "stat": статичные (кластерные) выбросы,
      - "lin" : линейные выбросы (ложная линейная тенденция),
      - "rand": случайные выбросы (точки, распределённые равномерно).
    n_out - количество групп выбросов,
    k - параметр для расчёта числа точек в группе.
    """
    conic_type = generate_random_conic_type()
    noise_std = np.random.uniform(4, 8)
    N = 1000
    x_range = (0, 100)
    
    if conic_type == "ellipse":
        x_clean, y_clean, x_noisy, y_noisy, params = generate_ellipse_data(noise_std, N, x_range)
        # Параметры эллипса: cx, cy, a_axis, b_axis, theta.
        filename_base = f"e_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}_{params[3]:.2f}_{params[4]:.2f}"
    elif conic_type == "parabola":
        x_clean, y_clean, x_noisy, y_noisy, params = generate_parabola_data(noise_std, N, x_range)
        # Параметры параболы: h, k, A.
        filename_base = f"p_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.4f}"
    else:
        x_clean, y_clean, x_noisy, y_noisy, params = generate_hyperbola_data(noise_std, N, x_range)
        # Параметры гиперболы: h, k, a, b.
        filename_base = f"h_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}_{params[3]:.2f}"
    
    # Аккумулируем выбросы.
    outlier_x_list = []
    outlier_y_list = []
    
    if category == "stat":
        filename_base += "_s"
        # Статичные (кластерные) выбросы, как в предыдущей версии.
        for _ in range(n_out):
            rand_x = np.random.uniform(x_range[0], x_range[1])
            idx = np.argmin(np.abs(x_clean - rand_x))
            true_y = y_clean[idx]
            outlier_center_y = true_y + np.random.uniform(20, 50) * np.random.choice([-1, 1])
            outlier_center_x = rand_x
            outlier_std = np.random.uniform(2, 7)
            num_outliers = int((50*k)/(1-0.05*k))
            
            x_outliers, y_outliers = generate_outliers(num_outliers, outlier_center_x, outlier_center_y, outlier_std)
            outlier_x_list.append(x_outliers)
            outlier_y_list.append(y_outliers)
            filename_base += f"_{num_outliers}"
    
    elif category == "lin":
        filename_base += "_l"
        # Линейные выбросы: ложная линия с собственными параметрами.
        for _ in range(n_out):
            num_points = int(((50*k)/(1-0.05*k)) / n_out)
            a_out, b_out = generate_linear_outlier_line_params(x_range)
            x_out, y_out = generate_linear_outliers(num_points, a_out, b_out, x_range, noise_std)
            outlier_x_list.append(x_out)
            outlier_y_list.append(y_out)
            filename_base += f"_{num_points}"
    
    else:  # category == "rand"
        filename_base += "_r"
        # Random noise выбросы: точки, равномерно распределённые в области основных данных.
        y_max = np.max(y_clean)
        y_range = (-25, y_max + 25)
        num_points = int(((50*k)/(1-0.05*k)) / n_out)
        x_out, y_out = generate_random_noise_outliers(num_points, x_range, y_range)
        outlier_x_list.append(x_out)
        outlier_y_list.append(y_out)
        filename_base += f"_{num_points}"
    
    X = np.concatenate((x_noisy, *outlier_x_list))
    Y = np.concatenate((y_noisy, *outlier_y_list))
    
    csv_filename = f"data_outliers/{filename_base}.csv"
    plot_filename = f"graph_outliers/{filename_base}.png"
    
    save_data_to_csv(X, Y, csv_filename)
    
    plt.figure(figsize=(8, 6))
    plt.scatter(X, Y, color='gray', alpha=0.7, label='Данные с выбросами')
    plt.plot(x_clean, y_clean, color='red', linewidth=2, label='Чистая кривая')
    plt.title(f'{conic_type.capitalize()} с шумом')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    
    plt.savefig(plot_filename)
    plt.show()

for i in range(1, 6):
    for k in range(14):
        gen_conic_outlier(i, k, category="stat")

for i in range(1, 3):
    for k in range(14):
        gen_conic_outlier(i, k, category="lin")

for i in range(1, 3):
    for k in range(14):
        gen_conic_outlier(1, k, category="rand")


# gen_conic_outlier(1, 6, "lin")